# 数据与运行调试
默认使用明确标记的合成数据，无需联网即可逐格运行。选择项目 `py311` 内核。切换真实行情时先改 `DEMO=False`，再修改股票范围与更新开关。分钟数据仅下载扫描入选股和自选股。

In [ ]:
from selectstock.runtime import check_runtime
check_runtime()
from selectstock.progress import DownloadProgress
from pathlib import Path
from copy import deepcopy
import pandas as pd
from IPython.display import display
from selectstock.config import load_config, save_experiment
from selectstock.demo import create_demo_config
from selectstock.runner import update_data, run_scans, load_bars, export_watchlists, check_data
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "scanners.yaml").exists())

In [ ]:
# 集中参数区。空列表表示全市场，建议先用少量真实股票验证。
DEMO = True
SYMBOLS = ["AAPL", "NVDA", "TSLA"]
WATCHLIST = ["AAPL"]
HISTORY_DAYS = {"1d": 730, "15m": 30, "5m": 30, "1m": 7}
READ_BARS = 252
END = None
LEADER_ENABLED = True
POTENT_ENABLED = True
RUN_DAILY_UPDATE = False
RUN_INTRADAY_UPDATE = False
DISPLAY_SYMBOL = None
config = create_demo_config(ROOT) if DEMO else load_config(ROOT)
if not DEMO:
    config["data"].update(symbols=SYMBOLS, end=END, history_days=HISTORY_DAYS,
                          read_bars=READ_BARS, offline=False)
    config["watchlist"]["symbols"] = WATCHLIST
config["scanners"]["leader"]["enabled"] = LEADER_ENABLED
config["scanners"]["potent"]["enabled"] = POTENT_ENABLED
print("合成演示数据" if DEMO else "Yahoo 真实行情", config["data"]["database"])

In [ ]:
# 数据库检查：真实模式首次运行时可以先执行下一格初始化/更新。
if Path(config["data"]["database"]).exists():
    display(check_data(config))
else:
    print("尚无数据库：打开 RUN_DAILY_UPDATE 后运行下一格。")

In [ ]:
# 单独更新日线。已有完整区间从本地读取，网络失败可重新执行本格续传。
if RUN_DAILY_UPDATE and not DEMO:
    daily_config = deepcopy(config)
    daily_config["data"]["timeframes"] = ["1d"]
    check_runtime()
    download_report = update_data(daily_config, progress=DownloadProgress(notebook=True))
    display(download_report.to_dict())
else:
    print("日线更新开关已关闭，使用本地数据。")

In [ ]:
# Leader 可以独立运行/关闭；本周累计至最近收盘日。
leader_result = run_scans(config, enabled=["leader"]) if LEADER_ENABLED else None
if leader_result:
    display(leader_result.coverage)
    display(leader_result.selected("leader"))

In [ ]:
# Potent 可以独立运行/关闭。
potent_result = run_scans(config, enabled=["potent"]) if POTENT_ENABLED else None
if potent_result:
    display(potent_result.coverage)
    display(potent_result.selected("potent"))

In [ ]:
# 同次导出两份名单与合并名单，TXT 可用于 TradingView（演示名单为虚构代码）。
results = run_scans(config)
output = export_watchlists(results)
print("输出目录:", output)
for name in results.frames:
    display(results.frames[name][["symbol", "status", "reasons"]])

In [ ]:
# 按需更新入选股与自选股分钟线。关闭的扫描器不会提供下载名单。
if RUN_INTRADAY_UPDATE and not DEMO:
    minute_config = deepcopy(config)
    minute_config["data"]["timeframes"] = ["15m", "5m", "1m"]
    display(update_data(minute_config, progress=DownloadProgress(notebook=True)).to_dict())
else:
    print("分钟更新开关已关闭。")

In [ ]:
from selectstock.plots import plot_symbol
symbol = DISPLAY_SYMBOL or config["data"]["symbols"][0]
display(load_bars(symbol, "1d", end=config["data"]["end"], limit=10, config=config))
figure = plot_symbol(symbol, config, limit=150)
display(figure)
import matplotlib.pyplot as plt
plt.close(figure)

数据诊断中的 `insufficient` 表示资料不足，不是未通过策略条件。收盘价与成交量采用 Yahoo 不做股息复权的口径；分钟数据保留交易时段标签。全市场首次初始化较慢，可中断后续传。修改下载历史长度不删除本地旧数据。